<a href="https://colab.research.google.com/github/yenlung/AI-Demo/blob/master/%E3%80%90Demo02CNN%E3%80%91%E7%94%A8CNN%E5%81%9A%E6%89%8B%E5%AF%AB%E8%BE%A8%E8%AD%98.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 【Demo02】用 CNN 做手寫辨識

上一次我們用標準的全連接神經網路 (Dense NN) 做 MNIST 手寫辨識, 這次我們要玩個更厲害、而且更適合「看圖」的神經網路 — **CNN (Convolutional Neural Network, 卷積神經網路)**。

CNN 跟前面的神經網路差在哪? 最直覺的說法是:

* 以前的 NN: 把一張 28×28 的圖**拉平**成 784 個數字, 再丟進去學。
* CNN: 保留圖片**二維的樣子**, 讓電腦用「小放大鏡」一塊一塊去觀察, 找出筆畫、邊緣、局部圖形這類「在地特徵」。

這樣的好處是: CNN 會自己學會像人一樣看圖。我們這次就來一層一層打造它, 最後再用 Gradio 做成一個可以讓人在網頁上畫數字的 Web App!

## 1. 讀入套件

習慣上, 我們會把這次會用到的東西一次 import 進來。

In [ ]:
!pip install gradio

In [ ]:
%matplotlib inline

# 標準數據分析、畫圖套件
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# 神經網路方面
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import SGD

# 互動測試用
from ipywidgets import interact_manual

# 神速打造 web app 的 Gradio
import gradio as gr

## 2. 讀入 MNIST 數據集

MNIST 應該不陌生了, 就是 28×28 的手寫數字 0~9 的灰階圖。

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

In [ ]:
print('訓練資料:', x_train.shape)
print('測試資料:', x_test.shape)

## 3. 資料整理 (這次有小小不同!)

### 3.1 為什麼 CNN 需要告訴它「channel」?

以前做 Dense 網路, 我們會把 28×28 拉平成一條長向量 (784)。

可是 CNN 要保留「圖片的二維結構」, 所以它想看到的是**一張圖**, 不是一條線。而且它還希望你告訴它:

> 這張圖有幾個「顏色頻道」(channel) 呢?

* 彩色圖: 有 RGB 三個 channel → `(高, 寬, 3)`
* 灰階圖: 只有一個亮度 → `(高, 寬, 1)`

MNIST 是灰階, 所以每張圖的形狀要改成 `(28, 28, 1)`。

In [ ]:
x_train = x_train.reshape(60000, 28, 28, 1) / 255
x_test  = x_test.reshape(10000, 28, 28, 1) / 255

順便把像素值從 0~255 壓到 0~1, 這叫 **normalize**, 可以讓訓練更穩定。

### 3.2 輸出做 one-hot 編碼

因為我們要預測的是 0~9 十個類別, 一樣用 one-hot 表示比較方便。

In [ ]:
y_train = to_categorical(y_train, 10)
y_test  = to_categorical(y_test, 10)

## 4. 打造 CNN 之前, 先搞懂它在做什麼

在真的寫程式疊層之前, 先花 30 秒理解兩個關鍵角色:

### 4.1 卷積層 Conv2D — 「拿著小放大鏡到處掃」

Conv2D 會在圖上用一個小視窗 (常見 3×3) 滑來滑去, 每滑到一個位置, 就算一下這塊地方「像不像某個圖樣」。

這個小視窗我們叫它 **filter** (或 kernel), 每個 filter 專門偵測某一種特徵, 例如: 直線、橫線、斜線、轉角、圓弧...

寫成程式長這樣:

```python
Conv2D(16, (3, 3), padding='same', activation='relu')
```

拆開看:
* `16` → 我們放了 **16 個 filter**, 也就是一次找 16 種特徵。
* `(3, 3)` → 每個 filter 都是 3×3 的小方塊。
* `padding='same'` → 掃的時候邊邊補 0, 讓**輸出大小跟輸入一樣**, 比較好算。
* `activation='relu'` → 跟以前一樣, 加個激發函數讓它變非線性。

### 4.2 池化層 MaxPooling2D — 「縮圖但保留重點」

掃完之後, 我們通常會用 `MaxPooling2D(pool_size=(2, 2))` 把圖縮小一半。作法是把每 2×2 一小塊的最大值挑出來, 其它丟掉。

好處有兩個:
1. **資料量變小**, 運算變快。
2. **強調明顯的特徵**, 對一點點位移也比較不敏感。

### 4.3 CNN 的經典節奏

CNN 通常就是反覆:

> Conv2D → MaxPooling → Conv2D → MaxPooling → ...

然後 filter 的數量會**越疊越多** (例如 16 → 32 → 64), 因為:
* 淺層看的是**簡單特徵** (邊緣、筆畫), 不需要太多種。
* 深層看的是**複雜組合** (形狀、局部結構), 需要更多種 filter 才夠用。

好, 理論夠了, 來蓋!

## 5. Step 1: 打造我們的 CNN 函數學習機

老樣子, 先開一個空的 `Sequential` 神經網路, 然後一層一層疊進去。

In [ ]:
model = Sequential()

### 5.1 第一組: Conv + Pooling (16 個 filter)

第一層要特別加 `input_shape=(28, 28, 1)`, 告訴模型**第一張進來的圖長什麼樣**。之後的層它就自己算得出來了。

In [ ]:
model.add(Conv2D(16, (3,3), padding='same',
                 input_shape=(28,28,1),
                 activation='relu'))

掃完之後, 縮小一半: 28×28 → 14×14。

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

### 5.2 第二組: Conv + Pooling (32 個 filter)

到這一層, 我們想看**比較複雜的特徵**, 所以 filter 加倍到 32 個。

In [ ]:
model.add(Conv2D(32, (3,3), padding='same',
                 activation='relu'))

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

這次縮完是 14×14 → 7×7。

### 5.3 第三組: Conv + Pooling (64 個 filter)

再更深一層, filter 再加倍到 64 個, 看更抽象的組合特徵。

In [ ]:
model.add(Conv2D(64, (3,3), padding='same',
                 activation='relu'))

In [ ]:
model.add(MaxPooling2D(pool_size=(2,2)))

這一次 7×7 縮一半、除不盡怎麼辦? Keras 會自動幫我們變成 3×3 (向下取整)。

### 5.4 Flatten — 把立體的特徵壓成一條

經過三組 Conv+Pooling, 我們現在手上是一個 `(3, 3, 64)` 的立體特徵。但要做「最後分類」, 還是要交給全連接層 (`Dense`) , 所以要先把它**拉平**成一條向量:

$$3 \times 3 \times 64 = 576$$

In [ ]:
model.add(Flatten())

### 5.5 最後兩層 Dense — 做「綜合判斷」

前面 CNN 找到各種特徵, 現在要把這些特徵**綜合起來判斷這是哪個數字**。

先來一層 64 個神經元的隱藏層:

In [ ]:
model.add(Dense(64, activation='relu'))

最後輸出層 10 個神經元, 對應 0~9。用 `softmax` 讓 10 個輸出加起來等於 1, 就像機率一樣!

In [ ]:
model.add(Dense(10, activation='softmax'))

## 6. 檢視一下我們的神經網路

來看看剛剛蓋出來的 CNN 長什麼樣, 順便觀察每一層的 output shape 是不是跟我們剛剛推的一樣 (28→14→7→3)。

In [ ]:
model.summary()

**小練習**: 看看總共有多少個參數? 跟你之前做過的 Dense 網路比, 是多是少? 想一想為什麼 CNN 參數常常比 Dense 少 — 因為 filter 是**共享**的, 同一個 3×3 小窗口掃整張圖, 不管位置都用同一組數字!

## 7. 組裝 (compile)

跟 Demo01 一樣, 要告訴模型:
* 用什麼 **loss function** 量衡錯多少
* 用什麼 **optimizer** 來調整參數
* 我們還想看什麼 **metric** (這裡看正確率)

In [ ]:
model.compile(loss='mse',
              optimizer=SGD(learning_rate=0.087),
              metrics=['accuracy'])

**修改建議**: 其實分類問題更常用的 loss 是 `categorical_crossentropy`, optimizer 換 `'adam'` 會更快收斂。等下訓練完你可以自己試試看差多少!

## 8. Step 2: 訓練 (fit)

CNN 比 Dense 慢一點, 但配合 GPU 還是非常快。我們跑 12 個 epochs 看看。

In [ ]:
model.fit(x_train, y_train, batch_size=128, epochs=12)

## 9. Step 3: 用測試資料來驗收

### 9.1 測試成績

用沒看過的測試資料來評估一下我們的 CNN 程度如何。

In [ ]:
loss, acc = model.evaluate(x_test, y_test)

In [ ]:
print(f'測試資料的正確率為 {acc*100:.2f}%')

### 9.2 抽幾張來看

先把所有預測結果算出來, 再用 `interact_manual` 做互動, 可以自己拉 slider 看第幾張圖被認成什麼。

In [ ]:
y_predict = np.argmax(model.predict(x_test), axis=-1)

In [ ]:
def my_predict(n):
    print('我可愛的 CNN 預測是', y_predict[n])
    X = x_test[n].reshape(28,28)
    plt.imshow(X, cmap='Greys')

In [ ]:
interact_manual(my_predict, n=(0, 9999));

## 10. 用 Gradio 做成手寫板 Web App!

光在 notebook 測試沒什麼感覺, 我們來做個**真的讓人在網頁上畫、CNN 當場判讀**的小 App。

### 10.1 把使用者畫的圖變成模型吃得下的樣子

使用者在 Gradio 畫板上畫的圖, 是一張 RGBA 的彩色圖, 而且通常比 28×28 大。我們要:

1. 把透明通道貼到白色背景上 (變成白底)
2. 轉成灰階
3. 縮小到 28×28
4. 黑白反轉 (因為 MNIST 是**黑底白字**, 畫板是白底黑字)
5. normalize 到 0~1
6. **reshape 成 `(1, 28, 28, 1)`** — CNN 要 channel!

In [ ]:
def resize_image(inp):
    # Sketchpad 把畫出來的圖放在 inp["layers"][0]
    image = np.array(inp["layers"][0], dtype=np.float32).astype(np.uint8)

    # 轉成 PIL 圖像
    image_pil = Image.fromarray(image)

    # 把透明通道貼到白色背景上, 再轉 RGB
    background = Image.new("RGB", image_pil.size, (255, 255, 255))
    background.paste(image_pil, mask=image_pil.split()[3])
    image_pil = background

    # 轉灰階
    image_gray = image_pil.convert("L")

    # 縮小成 28x28
    img_array = np.array(image_gray.resize((28, 28), resample=Image.LANCZOS))

    # MNIST 是黑底白字, 畫板是白底黑字, 要反轉
    img_array = 255 - img_array

    # 這裡跟 Dense 版不一樣! CNN 要 (1, 28, 28, 1)
    img_array = img_array.reshape(1, 28, 28, 1) / 255.0

    return img_array

### 10.2 包一個丟給模型預測的函式

輸出我們回傳「每個數字的可能性」, Gradio 就會自動幫我們畫機率長條圖。

In [ ]:
def recognize_digit(inp):
    img_array = resize_image(inp)
    prediction = model.predict(img_array).flatten()
    labels = list('0123456789')
    return {labels[i]: float(prediction[i]) for i in range(10)}

### 10.3 啟動 Web App!

`share=True` 會產生一個**可以丟給同學玩**的公開連結 (有效 72 小時)。

In [ ]:
iface = gr.Interface(
    fn=recognize_digit,
    inputs=gr.Sketchpad(),
    outputs=gr.Label(num_top_classes=3),
    title="我的 CNN 手寫辨識",
    description="請在畫板上用滑鼠畫一個 0~9 的數字, 看看我的 CNN 認不認得!"
)

iface.launch(share=True, debug=True)

恭喜! 你剛剛完成了一個可以被真人在瀏覽器上使用的 CNN 手寫辨識 Web App!

**接下來你可以試試看**:
1. 把 `loss` 換成 `'categorical_crossentropy'`, optimizer 換成 `'adam'`, 看看正確率會不會更高。
2. 把中間的 Conv 層加深一點, 或改 filter 數量 (例如 32 → 64 → 128)。
3. 加入 `Dropout(0.25)` 層看看能不能防止 overfitting。
4. 把 kernel size 改成 `(5, 5)` 看會不會比較準。

每改一個地方, 觀察 `model.summary()` 和訓練結果, 就會越來越有感覺!

## 11. 把訓練好的 model 存起來

如果你是在 Colab 上跑, 可以把模型存到自己的 Google Drive, 下次直接 load 就不用重新訓練了。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd '/content/drive/My Drive/Colab Notebooks'

In [ ]:
model.save('my_cnn_model.keras')

下次要用的時候, 只要:

```python
from tensorflow.keras.models import load_model
model = load_model('my_cnn_model.keras')
```

就可以把你訓練好的 CNN 叫回來了!